# 02 - Data Preprocessing

This notebook reads the raw crime dataset produced by the Stage 2 MVP collection workflow.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = "/content/drive/MyDrive/Projects/london_safety_analysis"
RAW_DIR = f"{PROJECT_ROOT}/data/raw"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

RAW_CRIME_FILE = os.path.join(RAW_DIR, "crime_data_raw.csv")
print("Reading:", RAW_CRIME_FILE)


In [ ]:
crime_data = pd.read_csv(RAW_CRIME_FILE)
print(f"Loaded {len(crime_data):,} raw crime rows")
print("Columns:", list(crime_data.columns))
crime_data.head()


## Basic cleaning


In [ ]:
crime_data_clean = crime_data.drop_duplicates().copy()

critical_cols = ["category", "borough", "collection_month"]
crime_data_clean = crime_data_clean.dropna(subset=critical_cols)

crime_data_clean["month"] = pd.to_datetime(crime_data_clean["collection_month"])
crime_data_clean["year"] = crime_data_clean["month"].dt.year
crime_data_clean["month_name"] = crime_data_clean["month"].dt.month_name()

crime_category_mapping = {
    "violent-crime": "Violent Crime",
    "violence-and-sexual-offences": "Violent Crime",
    "criminal-damage-arson": "Property Crime",
    "burglary": "Property Crime",
    "vehicle-crime": "Property Crime",
    "theft-from-the-person": "Property Crime",
    "shoplifting": "Property Crime",
    "other-theft": "Property Crime",
    "bicycle-theft": "Property Crime",
    "drugs": "Drug Crime",
    "possession-of-weapons": "Weapon Crime",
    "public-order": "Public Order",
    "anti-social-behaviour": "Anti-Social Behaviour",
    "other-crime": "Other Crime",
    "robbery": "Violent Crime",
}
crime_data_clean["crime_type"] = crime_data_clean["category"].map(crime_category_mapping).fillna("Other Crime")

severity_scores = {
    "Violent Crime": 5,
    "Weapon Crime": 5,
    "Drug Crime": 4,
    "Property Crime": 3,
    "Public Order": 2,
    "Anti-Social Behaviour": 1,
    "Other Crime": 2,
}
crime_data_clean["severity_score"] = crime_data_clean["crime_type"].map(severity_scores)

print(f"Rows after cleaning: {len(crime_data_clean):,}")
crime_data_clean.head()


## Save Stage 2 preprocessing outputs


In [ ]:
borough_stats = crime_data_clean.groupby("borough").agg(
    total_crimes=("category", "count"),
    avg_severity=("severity_score", "mean")
).reset_index()

monthly_trends = crime_data_clean.groupby(["month", "crime_type"]).size().unstack(fill_value=0)

crime_data_clean.to_csv(os.path.join(PROCESSED_DIR, "crime_data_processed.csv"), index=False)
borough_stats.to_csv(os.path.join(PROCESSED_DIR, "borough_statistics.csv"), index=False)
monthly_trends.to_csv(os.path.join(PROCESSED_DIR, "monthly_trends.csv"))

print("Saved processed outputs to:", PROCESSED_DIR)
